In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from catboost import Pool, CatBoostClassifier, CatBoostRegressor, CatBoostRanker
from sklearn.metrics import roc_auc_score
from sklearn.metrics import average_precision_score, precision_recall_curve, auc

In [2]:
data_path = '/media/dm/1TB/Data Fusion Contest 2026/task1/'

In [3]:
%%time
#экономия ОЗУ
train = pl.concat([
        pl.scan_parquet(data_path+'pretrain_part_*.parquet'
                          ).with_columns(pl.lit(True).alias("pretrain"),
                                         pl.col("session_id").cast(pl.Int64)
                                        ),
        pl.scan_parquet(data_path+'train_part_*.parquet'
                          ).with_columns(pl.lit(False).alias("pretrain")),
                   ]
                          ).with_columns(pl.col('event_dttm').str.to_datetime(),
                                         
                                         pl.col("event_type_nm").cast(pl.Int8),
                                         pl.col("event_desc").cast(pl.UInt8),
                                         pl.col("channel_indicator_type").cast(pl.Int8),
                                         pl.col("channel_indicator_sub_type").cast(pl.Int8),
                                         pl.col("currency_iso_cd").cast(pl.Int8),
                                         #pl.col("operaton_amt").log1p().cast(pl.Float32),
                                         pl.col("mcc_code").cast(pl.Int8),
                                         pl.col("pos_cd").cast(pl.Int8),
                                         pl.col("timezone").cast(pl.UInt16),
                                         pl.col("operating_system_type").cast(pl.Int8),
                                         pl.col("developer_tools").cast(pl.Int8),
                                         pl.col("phone_voip_call_state").cast(pl.Int8),
                                         pl.col("web_rdp_connection").cast(pl.Int8),
                                         pl.col("compromised").cast(pl.Int8),
                                         #pl.col("device_system_version").cast(pl.Int8),
                                         pl.col("device_system_version").is_null(),
                                         #pl.col("battery").fill_null("-99%").str.replace("not available","-90%"
                                         #                ).str.split("%").list.first().str.replace_all(":","").cast(pl.Float32).fill_nan(105).round().cast(pl.Int8),
                                         pl.col("battery").is_null(),
                                         pl.col("browser_language").is_null(),
                                         pl.col("screen_size").str.replace_all(" ","").str.split("x"),
                          ).with_columns(
                                         pl.col("screen_size").list.first().cast(pl.Float32).alias("screen_w"),
                                         pl.col("screen_size").list.last().cast(pl.Float32).alias("screen_h"),
                          ).with_columns( (pl.col('screen_w')/pl.col('screen_h')*40).round().cast(pl.UInt8).alias("screen_"),
                                        ).drop("screen_size","accept_language"
                          ).sort("customer_id","event_dttm"
               ).fill_null(-1
               ).fill_null("null"
                                ).collect()
train

CPU times: user 2min 57s, sys: 50.5 s, total: 3min 47s
Wall time: 1min 31s


customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_
i64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16
123123123123129,123251972261925,2023-10-01 09:16:41,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,125193298164858,2023-10-01 11:13:48,14,75,6,5,71085.0,0,15,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,124823932244729,2023-10-01 11:40:35,14,75,6,5,36158.0,0,14,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,124454563399321,2023-10-01 16:37:32,14,75,6,5,81971.0,0,4,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,126490377886229,2023-10-02 19:28:25,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
124265584425461,125416637435562,2025-05-24 06:48:12,7,56,4,15,-1.0,-1,-1,-1,true,-1,126052290873165,-1,true,false,0,0,-1,0,false,1080.0,2168.0,20
124265584425461,123827498403451,2025-05-24 06:50:19,7,56,4,15,-1.0,-1,-1,-1,true,-1,124514693381291,-1,true,false,0,0,-1,0,false,1080.0,2168.0,20
124265584425461,125330740018955,2025-05-24 06:50:41,14,51,4,15,361728.0,0,-1,-1,true,-1,124514693381291,-1,true,false,0,0,-1,0,false,1080.0,2168.0,20


In [6]:
cols = [
#"event_dttm",
"event_type_nm",
"event_desc",
"channel_indicator_type",
"channel_indicator_sub_type",
"operaton_amt",
"currency_iso_cd",
"mcc_code",
"pos_cd",
#"accept_language",
"browser_language",
"timezone",
"session_id",
"operating_system_type",
"battery",
"device_system_version",
#"screen_size",
"developer_tools",
"phone_voip_call_state",
"web_rdp_connection",
"compromised"
]

In [4]:
%%time
#Генерация свойств по предистории 
train0 = train.with_columns(
                   (pl.col("event_dttm")-pl.col("event_dttm").shift(1).over("customer_id")).dt.total_seconds().alias("pause_cus"),
                   (pl.col("event_dttm")-pl.col("event_dttm").shift(1).over("session_id")).dt.total_seconds().alias("pause_ses"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id").alias("operaton_amt_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","mcc_code").alias("operaton_amt_mcc_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","channel_indicator_type").alias("operaton_amt_type_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","event_desc").alias("operaton_amt_desc_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","channel_indicator_sub_type").alias("operaton_amt_sub_max_prev"),
                   pl.lit(1).alias("one") 
                  )

for col in cols:
    print(col, end=" ")
    train0 = train0.with_columns(
                                 (pl.col(col) == pl.col(col).shift(1).over("customer_id")).alias(f"{col}_prev"),
                                 pl.col("one").rolling_sum(window_size=9999999,min_samples=1).log1p().cast(pl.Float32).over("customer_id",col).alias(f"{col}_log_count"),
                                )
train0 = train0.with_columns(pl.col("one").rolling_sum(window_size=9999999,min_samples=0
                                                      ).cast(pl.UInt16).over("customer_id","session_id").alias(f"{col}_ses_count"),
                            )
train0

event_type_nm event_desc channel_indicator_type channel_indicator_sub_type operaton_amt currency_iso_cd mcc_code pos_cd browser_language timezone session_id operating_system_type battery device_system_version developer_tools phone_voip_call_state web_rdp_connection compromised CPU times: user 17min 53s, sys: 56.5 s, total: 18min 49s
Wall time: 5min 59s


customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_,pause_cus,pause_ses,operaton_amt_max_prev,operaton_amt_mcc_max_prev,operaton_amt_type_max_prev,operaton_amt_desc_max_prev,operaton_amt_sub_max_prev,one,event_type_nm_prev,event_type_nm_log_count,event_desc_prev,event_desc_log_count,channel_indicator_type_prev,channel_indicator_type_log_count,channel_indicator_sub_type_prev,channel_indicator_sub_type_log_count,operaton_amt_prev,operaton_amt_log_count,currency_iso_cd_prev,currency_iso_cd_log_count,mcc_code_prev,mcc_code_log_count,pos_cd_prev,pos_cd_log_count,browser_language_prev,browser_language_log_count,timezone_prev,timezone_log_count,session_id_prev,session_id_log_count,operating_system_type_prev,operating_system_type_log_count,battery_prev,battery_log_count,device_system_version_prev,device_system_version_log_count,developer_tools_prev,developer_tools_log_count,phone_voip_call_state_prev,phone_voip_call_state_log_count,web_rdp_connection_prev,web_rdp_connection_log_count,compromised_prev,compromised_log_count,compromised_ses_count
i64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16,i64,i64,f64,f64,f64,f64,f64,i32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,u16
123123123123129,123251972261925,2023-10-01 09:16:41,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,null,null,-1.0,-1.0,-1.0,-1.0,-1.0,1,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,1
123123123123129,125193298164858,2023-10-01 11:13:48,14,75,6,5,71085.0,0,15,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,7027,7027,71085.0,71085.0,71085.0,71085.0,71085.0,1,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,2
123123123123129,124823932244729,2023-10-01 11:40:35,14,75,6,5,36158.0,0,14,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,1607,1607,71085.0,36158.0,71085.0,71085.0,71085.0,1,true,1.098612,true,1.098612,true,1.098612,true,1.098612,false,0.693147,true,1.098612,false,0.693147,true,1.098612,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,3
123123123123129,124454563399321,2023-10-01 16:37:32,14,75,6,5,81971.0,0,4,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,17817,17817,81971.0,81971.0,81971.0,81971.0,81971.0,1,true,1.386294,true,1.386294,true,1.386294,true,1.386294,false,0.693147,true,1.386294,false,0.693147,true,1.386294,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,4
123123123123129,126490377886229,2023-10-02 19:28:25,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,96653,96653,81971.0,-1.0,-1.0,-1.0,-1.0,1,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
124265584425461,125416637435562,2025

In [5]:
train0.write_parquet("train0.pq")
train0.shape

(176617797, 70)

In [6]:
#train0 = pl.read_parquet("train0.pq")
#train0.shape

In [7]:
%%time
#наклейка меток
labels = pl.scan_parquet(data_path+'train_labels.parquet'
                        ).with_columns(pl.col('target').cast(pl.Boolean),
                        ).collect(
         ).join(train0.filter(~pl.col("pretrain")
                             ).drop("pretrain","customer_id"),
                on=[#"customer_id",
                    "event_id"],
                how="right"
        ).with_columns( pl.col("target").is_not_null().alias("label")
               ).fill_null(False
)
labels
#shape: (87_514, 3)
#shape: (85_677_840, 49)

CPU times: user 14.9 s, sys: 15.4 s, total: 30.2 s
Wall time: 5.35 s


customer_id,target,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,screen_w,screen_h,screen_,pause_cus,pause_ses,operaton_amt_max_prev,operaton_amt_mcc_max_prev,operaton_amt_type_max_prev,operaton_amt_desc_max_prev,operaton_amt_sub_max_prev,one,event_type_nm_prev,event_type_nm_log_count,event_desc_prev,event_desc_log_count,channel_indicator_type_prev,channel_indicator_type_log_count,channel_indicator_sub_type_prev,channel_indicator_sub_type_log_count,operaton_amt_prev,operaton_amt_log_count,currency_iso_cd_prev,currency_iso_cd_log_count,mcc_code_prev,mcc_code_log_count,pos_cd_prev,pos_cd_log_count,browser_language_prev,browser_language_log_count,timezone_prev,timezone_log_count,session_id_prev,session_id_log_count,operating_system_type_prev,operating_system_type_log_count,battery_prev,battery_log_count,device_system_version_prev,device_system_version_log_count,developer_tools_prev,developer_tools_log_count,phone_voip_call_state_prev,phone_voip_call_state_log_count,web_rdp_connection_prev,web_rdp_connection_log_count,compromised_prev,compromised_log_count,compromised_ses_count,label
i64,bool,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,f32,f32,i16,i64,i64,f64,f64,f64,f64,f64,i32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,u16,bool
null,false,123999300382879,2024-10-01 05:29:14,14,75,6,5,56422.0,0,4,3,true,-1,-1,-1,true,true,-1,-1,-1,-1,-1.0,-1.0,-1,62061,62061,7.9752e7,652201.0,4.25935e7,4.25935e7,4.25935e7,1,true,6.692084,true,6.395262,true,6.583409,true,6.549651,false,0.693147,true,6.914731,false,5.777652,true,6.161207,true,7.323831,true,7.021084,true,7.323831,true,7.323831,true,7.323831,true,7.323831,true,7.323831,true,7.323831,true,7.323831,true,7.323831,1515,false
null,false,124531875713936,2024-10-01 10:17:22,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,-1.0,-1.0,-1,17288,17288,7.9752e7,7.9752e7,2.9739e7,-1.0,2.9739e7,1,false,6.206576,false,6.206576,false,5.983936,false,5.983936,false,6.298949,false,6.23637,false,6.678342,false,6.678342,true,7.32449,true,7.021976,true,7.32449,true,7.32449,true,7.32449,true,7.32449,true,7.32449,true,7.32449,true,7.32449,true,7.32449,1516,false
null,false,123329285580171,2024-10-01 10:20:03,3,120,6,5,300870.0,0,10,3,true,-1,-1,-1,true,true,-1,-1,-1,-1,-1.0,-1.0,-1,161,161,7.9752e7,1.2595e7,4.25935e7,1.2595e7,4.25935e7,1,false,3.89182,false,3.89182,false,6.584791,false,6.55108,false,0.693147,false,6.915723,false,4.844187,false,6.163315,true,7.325149,true,7.022868,true,7.325149,true,7.325149,true,7.325149,true,7.325149,true,7.325149,true,7.325149,true,7.325149,true,7.325149,1517,false
null,false,124334305430665,2024-10-02 07:48:09,14,75,6,5,298458.0,0,1,3,true,-1,-1,-1,true,true,-1,-1,-1,-1,-1.0,-1.0,-1,77286,77286,7.9752e7,4.25935e7,4.25935e7,4.25935e7,4.25935e7,1,false,6.693324,false,6.39693,true,6.586172,true,6.552508,false,0.693147,true,6.916715,false,3.806663,true,6.165418,true,7.325808,true,7.023759,true,7.325808,true,7.325808,true,7.325808,true,7.325808,true,7.325808,true,7.325808,true,7.325808,true,7.325808,1518,false
null,false,126215501146513,2024-10-02 11:20:40,14,75,6,5,59944.0,0,15,3,true,-1,-1,-1,true,true,-1,-1,-1,-1,-1.0,-1.0,-1,12751,12751,7.9752e7,297180.0,4.25935e7,4.25935e7,4.25935e7,1,true,6.694562,true,6.398595,true,6.58755,true,6.553934,false,0.693147,true,6.917706,false,4.990433,true,6.167517,true,7.326466,true,7.024649,true,7.326466,true,7.326466,true,7.326466,true,7.326466,true,7.326466,true,7.326466,true,7.326466,true,7.326466,1519,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…

In [8]:
cat_feature_names = []

In [9]:
%%time
limit=round( labels.shape[0]*0.6 )

###если тренировать только на меченом
#train_pool = Pool(data = labels[:limit].filter(pl.col("label")).drop(#"weight", 
###если тренировать на всем трейне
train_pool = Pool(data = labels[:limit].drop(#"weight", #если тренировать на всем трейне
                                             "customer_id","target","event_id","event_dttm","session_id","label"
                                      ).to_pandas(), 
###если тренировать только на меченом
#                      label = labels[:limit].filter(pl.col("label")).select("target"
###если тренировать на всем трейне
                      label = labels[:limit].select("target"
                                           ).to_pandas(), 
                      cat_features = cat_feature_names
                     )

###оценивать только на меченом
valid_pool = Pool(data = labels[limit:].filter(pl.col("label")
                                      ).drop(#"weight",
                                             "customer_id","target","event_id","event_dttm","session_id","label"
                                      ).to_pandas(), 
                      label = labels[limit:].filter(pl.col("label")
                                      ).select("target"
                                      ).to_pandas(), 
                      cat_features = cat_feature_names
                     )
valid_pool

CPU times: user 8.91 s, sys: 30.9 s, total: 39.8 s
Wall time: 6.48 s


In [10]:
model = CatBoostClassifier(
                            depth=7,
                           loss_function = 'Logloss',
                           #loss_function = 'Focal:focal_alpha=0.5;focal_gamma=0.1',
                           custom_metric=["Accuracy",'Logloss',"Precision","Recall",
                                          "F1","AUC","PRAUC"],
    
                           eval_metric="PRAUC",
                           #eval_metric=MyMetric(),
                           random_seed = 1234,
    
                           task_type="GPU",
                           verbose = 100)

In [11]:
%%time

model.fit(train_pool,
              plot=True,
              eval_set=valid_pool,
              early_stopping_rounds=200,
              verbose = 1000
             )
model.save_model(f"model00.cbm")
print(model.get_feature_importance(prettified=True))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Learning rate set to 0.02832


Default metric period is 5 because AUC, PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0


0:	learn: 0.0055365	test: 0.5984335	best: 0.5984335 (0)	total: 2.31s	remaining: 38m 28s


Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Setting Precision metric value to the default 0
Number of the positive class predictions is 0. Set

999:	learn: 0.0279821	test: 0.7368839	best: 0.7368839 (999)	total: 27m 27s	remaining: 0us
bestTest = 0.7368838602
bestIteration = 999
                    Feature Id  Importances
0                 operaton_amt    11.854271
1             pos_cd_log_count     7.729910
2                    pause_cus     6.761138
3                     mcc_code     5.992036
4                   event_desc     5.756007
..                         ...          ...
60     web_rdp_connection_prev     0.035497
61               timezone_prev     0.024503
62       browser_language_prev     0.022477
63  device_system_version_prev     0.016751
64                         one     0.000000

[65 rows x 2 columns]
CPU times: user 30min 42s, sys: 5min 17s, total: 36min
Wall time: 27min 39s


In [12]:
model.get_feature_importance(prettified=True).head(60)

,Feature Id,Importances
0,operaton_amt,11.854271
1,pos_cd_log_count,7.729910
2,pause_cus,6.761138
3,mcc_code,5.992036
4,event_desc,5.756007
5,event_desc_log_count,5.281206
6,pause_ses,4.929393
7,mcc_code_log_count,4.222202
8,operaton_amt_desc_max_prev,4.163511
9,event_type_nm,2.721051


In [13]:
average_precision_score(labels[limit:].filter(pl.col("label")
                                      ).select("target"
                                      ).to_pandas(),
                        model.predict_proba(valid_pool)[:,1]
                       )

0.7369119110755996

In [14]:
precisions, recalls, _ = precision_recall_curve(labels[limit:].filter(pl.col("label")
                                      ).select("target"
                                      ).to_pandas(),
                        model.predict_proba(valid_pool)[:,0])
auc(recalls, precisions)

0.4738131745665183

In [3]:
%%time
test = pl.concat([
        pl.scan_parquet(data_path+'pretrain_part_*.parquet'
                          ).with_columns(pl.lit(True).alias("pretrain"),
                                         pl.col("session_id").cast(pl.Int64)
                                        ),
        pl.scan_parquet(data_path+'train_part_*.parquet'
                          ).with_columns(pl.lit(True).alias("pretrain")),    
    
        pl.scan_parquet(data_path+'pretest.parquet'
                          ).with_columns(pl.lit(True).alias("pretrain"),
                                         pl.col("session_id").cast(pl.Int64)
                                        ),
        pl.scan_parquet(data_path+'test.parquet'
                          ).with_columns(pl.lit(False).alias("pretrain")),
                   ]
                          ).with_columns(pl.col('event_dttm').str.to_datetime(),
                                         
                                         pl.col("event_type_nm").cast(pl.Int8),
                                         pl.col("event_desc").cast(pl.UInt8),
                                         pl.col("channel_indicator_type").cast(pl.Int8),
                                         pl.col("channel_indicator_sub_type").cast(pl.Int8),
                                         pl.col("currency_iso_cd").cast(pl.Int8),
                                         #pl.col("operaton_amt").log1p().cast(pl.Float32),
                                         pl.col("mcc_code").cast(pl.Int8),
                                         pl.col("pos_cd").cast(pl.Int8),
                                         pl.col("timezone").cast(pl.UInt16),
                                         pl.col("operating_system_type").cast(pl.Int8),
                                         pl.col("developer_tools").cast(pl.Int8),
                                         pl.col("phone_voip_call_state").cast(pl.Int8),
                                         pl.col("web_rdp_connection").cast(pl.Int8),
                                         pl.col("compromised").cast(pl.Int8),
                                         pl.col("battery").is_null(),
                                         pl.col("device_system_version").is_null(),
                                         pl.col("browser_language").is_null(),
                                         pl.col("screen_size").str.replace_all(" ","").str.split("x"),
                          ).with_columns(
                                         pl.col("screen_size").list.first().cast(pl.Float32).alias("screen_w"),
                                         pl.col("screen_size").list.last().cast(pl.Float32).alias("screen_h"),
                          ).with_columns( (pl.col('screen_w')/pl.col('screen_h')*40).round().cast(pl.Int16).alias("screen_"),
                                        ).drop("screen_size","accept_language"
                        ).sort("customer_id","event_id"
               ).fill_null(-1
               ).fill_null("null"
                                ).collect()
test
#shape: (14_835_758, 24)
#shape: (191_453_555, 24)

CPU times: user 3min 21s, sys: 52.7 s, total: 4min 13s
Wall time: 1min 50s


customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_
i64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16
123123123123129,123123123584584,2024-01-03 12:37:20,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,123123125128477,2025-01-30 14:25:45,7,56,4,15,-1.0,-1,-1,-1,true,-1,123801728164416,-1,true,false,0,-1,-1,0,true,1080.0,2180.0,20
123123123123129,123123126851900,2024-10-20 16:28:40,14,75,0,5,49890.0,0,4,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,123123127456830,2025-06-26 05:46:09,14,75,0,5,15000.0,0,15,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,123131713680603,2024-10-14 16:51:11,14,75,0,5,101222.0,0,4,3,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
124265584425461,126541918174987,2024-05-29 07:43:38,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
124265584425461,126541918261472,2024-02-23 16:12:17,7,56,3,4,-1.0,-1,-1,-1,true,75,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
124265584425461,126541918479371,2024-06-29 19:17:13,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1


In [4]:
%%time
#удаление дублей по event_id
test = test.with_columns( (pl.col("event_id")==pl.col("event_id").shift(1)).alias("double")
                 ).filter(pl.col("double")==False
                         ).drop("double"
                               ).sort("customer_id","event_dttm"
                                     )
test
#shape: (191_310_789, 24)

CPU times: user 1min 17s, sys: 8.52 s, total: 1min 26s
Wall time: 17.3 s


customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_
i64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16
123123123123129,123251972261925,2023-10-01 09:16:41,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,125193298164858,2023-10-01 11:13:48,14,75,6,5,71085.0,0,15,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,124823932244729,2023-10-01 11:40:35,14,75,6,5,36158.0,0,14,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,124454563399321,2023-10-01 16:37:32,14,75,6,5,81971.0,0,4,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
123123123123129,126490377886229,2023-10-02 19:28:25,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
124265584425461,125932035970896,2025-06-23 04:43:35,7,56,4,15,-1.0,-1,-1,-1,true,-1,123586980267840,-1,true,false,0,0,-1,0,true,1080.0,2168.0,20
124265584425461,126267042326129,2025-06-26 05:15:27,7,56,4,15,-1.0,-1,-1,-1,true,-1,125562665697080,-1,true,false,0,0,-1,0,true,1080.0,2168.0,20
124265584425461,123870451172166,2025-06-26 05:16:37,7,56,4,15,-1.0,-1,-1,-1,true,-1,126481788436076,-1,true,false,0,0,-1,0,true,1080.0,2168.0,20


In [7]:
%%time
test0 = test.with_columns(
                   (pl.col("event_dttm")-pl.col("event_dttm").shift(1).over("customer_id")).dt.total_seconds().alias("pause_cus"),
                   (pl.col("event_dttm")-pl.col("event_dttm").shift(1).over("session_id")).dt.total_seconds().alias("pause_ses"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id").alias("operaton_amt_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","mcc_code").alias("operaton_amt_mcc_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","channel_indicator_type").alias("operaton_amt_type_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","event_desc").alias("operaton_amt_desc_max_prev"),
                   pl.col("operaton_amt").rolling_max(window_size=9999999,min_samples=1).over("customer_id","channel_indicator_sub_type").alias("operaton_amt_sub_max_prev"),
                   pl.lit(1).alias("one") 
                  )
for col in cols:
    print(col, end=" ")
    test0 = test0.with_columns(
                                 (pl.col(col) == pl.col(col).shift(1).over("customer_id")).alias(f"{col}_prev"),
                                 pl.col("one").rolling_sum(window_size=9999999,min_samples=1).log1p().cast(pl.Float32).over("customer_id",col).alias(f"{col}_log_count"),
                                )
test0 = test0.with_columns(pl.col("one").rolling_sum(window_size=9999999,min_samples=0
                                                      ).cast(pl.UInt16).over("customer_id","session_id").alias(f"{col}_ses_count"),
                            )
test0

event_type_nm event_desc channel_indicator_type channel_indicator_sub_type operaton_amt currency_iso_cd mcc_code pos_cd browser_language timezone session_id operating_system_type battery device_system_version developer_tools phone_voip_call_state web_rdp_connection compromised CPU times: user 19min 34s, sys: 1min 11s, total: 20min 45s
Wall time: 6min 40s


customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_,pause_cus,pause_ses,operaton_amt_max_prev,operaton_amt_mcc_max_prev,operaton_amt_type_max_prev,operaton_amt_desc_max_prev,operaton_amt_sub_max_prev,one,event_type_nm_prev,event_type_nm_log_count,event_desc_prev,event_desc_log_count,channel_indicator_type_prev,channel_indicator_type_log_count,channel_indicator_sub_type_prev,channel_indicator_sub_type_log_count,operaton_amt_prev,operaton_amt_log_count,currency_iso_cd_prev,currency_iso_cd_log_count,mcc_code_prev,mcc_code_log_count,pos_cd_prev,pos_cd_log_count,browser_language_prev,browser_language_log_count,timezone_prev,timezone_log_count,session_id_prev,session_id_log_count,operating_system_type_prev,operating_system_type_log_count,battery_prev,battery_log_count,device_system_version_prev,device_system_version_log_count,developer_tools_prev,developer_tools_log_count,phone_voip_call_state_prev,phone_voip_call_state_log_count,web_rdp_connection_prev,web_rdp_connection_log_count,compromised_prev,compromised_log_count,compromised_ses_count
i64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16,i64,i64,f64,f64,f64,f64,f64,i32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,u16
123123123123129,123251972261925,2023-10-01 09:16:41,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,null,null,-1.0,-1.0,-1.0,-1.0,-1.0,1,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,null,0.693147,1
123123123123129,125193298164858,2023-10-01 11:13:48,14,75,6,5,71085.0,0,15,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,7027,7027,71085.0,71085.0,71085.0,71085.0,71085.0,1,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,false,0.693147,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,true,1.098612,2
123123123123129,124823932244729,2023-10-01 11:40:35,14,75,6,5,36158.0,0,14,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,1607,1607,71085.0,36158.0,71085.0,71085.0,71085.0,1,true,1.098612,true,1.098612,true,1.098612,true,1.098612,false,0.693147,true,1.098612,false,0.693147,true,1.098612,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,true,1.386294,3
123123123123129,124454563399321,2023-10-01 16:37:32,14,75,6,5,81971.0,0,4,8,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,17817,17817,81971.0,81971.0,81971.0,81971.0,81971.0,1,true,1.386294,true,1.386294,true,1.386294,true,1.386294,false,0.693147,true,1.386294,false,0.693147,true,1.386294,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,true,1.609438,4
123123123123129,126490377886229,2023-10-02 19:28:25,7,56,4,15,-1.0,-1,-1,-1,true,-1,-1,-1,true,true,-1,-1,-1,-1,true,-1.0,-1.0,-1,96653,96653,81971.0,-1.0,-1.0,-1.0,-1.0,1,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,false,1.098612,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,true,1.791759,5
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
124265584425461,125932035970896,2025

In [8]:
%%time
submit = pl.read_csv(data_path+'sample_submit.csv')
submit
#shape: (633_683, 2)

CPU times: user 193 ms, sys: 29.7 ms, total: 223 ms
Wall time: 436 ms


event_id,predict
i64,f64
125854726334416,-0.984226
125949211749418,-1.14122
124437385134670,-0.869487
124394437682654,-0.869487
123973531121838,-1.039899
…,…
125304968700392,-1.196618
123930579661188,-0.869487
123140303897433,-0.869487


In [9]:
test0 = submit.join(test0, on="event_id"
                         )
test0
#shape: (633_683, 69)

event_id,predict,customer_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_,pause_cus,pause_ses,operaton_amt_max_prev,operaton_amt_mcc_max_prev,operaton_amt_type_max_prev,operaton_amt_desc_max_prev,operaton_amt_sub_max_prev,one,event_type_nm_prev,event_type_nm_log_count,event_desc_prev,event_desc_log_count,channel_indicator_type_prev,channel_indicator_type_log_count,channel_indicator_sub_type_prev,channel_indicator_sub_type_log_count,operaton_amt_prev,operaton_amt_log_count,currency_iso_cd_prev,currency_iso_cd_log_count,mcc_code_prev,mcc_code_log_count,pos_cd_prev,pos_cd_log_count,browser_language_prev,browser_language_log_count,timezone_prev,timezone_log_count,session_id_prev,session_id_log_count,operating_system_type_prev,operating_system_type_log_count,battery_prev,battery_log_count,device_system_version_prev,device_system_version_log_count,developer_tools_prev,developer_tools_log_count,phone_voip_call_state_prev,phone_voip_call_state_log_count,web_rdp_connection_prev,web_rdp_connection_log_count,compromised_prev,compromised_log_count,compromised_ses_count
i64,f64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16,i64,i64,f64,f64,f64,f64,f64,i32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,u16
123707242230467,-1.20836,123123123123129,2025-07-27 14:55:11,8,122,2,0,-1.0,-1,-1,-1,true,-1,125597024902696,-1,true,true,-1,-1,-1,-1,false,-1.0,-1.0,-1,94053,null,1.10227007e8,1.10227007e8,-1.0,-1.0,-1.0,1,false,5.509388,false,4.356709,false,4.356709,false,4.356709,false,6.943122,false,6.910751,false,7.30586,true,7.732808,true,8.018296,true,7.878534,false,0.693147,true,8.018296,true,8.018296,true,7.86442,true,7.86442,true,7.908755,true,7.998672,true,7.824846,1
123234793229123,-1.409739,123123123123129,2025-07-27 14:55:20,14,42,4,15,247750.0,0,-1,-1,true,-1,125597024902696,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,9,9,1.10227007e8,1.10227007e8,2.9739e7,2.012582e6,2.9739e7,1,false,7.466799,false,3.135494,false,6.855409,false,6.855409,false,0.693147,false,7.618251,true,7.306531,true,7.733246,true,8.018625,true,7.878913,true,1.098612,true,8.018625,true,8.018625,false,6.075346,false,6.075346,false,5.758902,true,7.999007,false,6.284134,2
125837545055907,-1.409739,123123123123129,2025-07-27 14:55:28,7,56,4,15,-1.0,-1,-1,-1,true,-1,125828952579121,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,8,null,1.10227007e8,1.10227007e8,2.9739e7,-1.0,2.9739e7,1,false,6.763885,false,6.766191,true,6.856462,true,6.856462,false,6.944087,false,6.911747,true,7.307202,true,7.733684,true,8.018954,true,7.879292,false,0.693147,true,8.018954,true,8.018954,true,6.077642,true,6.077642,true,5.762052,true,7.999343,true,6.285998,1
126456020999239,-1.20836,123123123123129,2025-07-27 14:56:28,7,56,4,15,-1.0,-1,-1,-1,true,-1,125597024902696,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,60,68,1.10227007e8,1.10227007e8,2.9739e7,-1.0,2.9739e7,1,true,6.765039,true,6.767343,true,6.857514,true,6.857514,true,6.945051,true,6.912743,true,7.307873,true,7.734121,true,8.019284,true,7.87967,false,1.386294,true,8.019284,true,8.019284,true,6.079933,true,6.079933,true,5.765191,true,7.999679,true,6.287858,3
125090221587312,-1.124658,123123123123130,2025-07-09 06:42:00,8,86,4,15,496350.0,0,-1,-1,true,-1,125605614487041,-1,true,false,0,0,-1,0,false,720.0,1507.0,19,106964,null,6.5867429e7,6.5867429e7,6.45125e7,2.9715e6,6.45125e7,1,false,6.09131,false,6.070738,true,7.597396,true,7.548029,false,0.693147,true,7.578146,true,7.866722,true,7.952967,true,8.083637,true,7.932721,false,0.693147,true,8.083637,true,8.083637,true,6.656726,true,6.656726,true,6.448889,t

In [10]:
%%time
pool = Pool(data = test0.drop("customer_id",
                                      "event_id","event_dttm","session_id"
                               ).fill_null(-1
                               ).fill_null("null"
                               ).to_pandas(), 
                     )

CPU times: user 102 ms, sys: 120 ms, total: 223 ms
Wall time: 66.7 ms


In [11]:
model = CatBoostClassifier()
model.load_model(f"model00.cbm")

In [12]:
test0 = test0.with_columns(predict =  model.predict_proba(pool)[:,1])
test0

event_id,predict,customer_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,pretrain,screen_w,screen_h,screen_,pause_cus,pause_ses,operaton_amt_max_prev,operaton_amt_mcc_max_prev,operaton_amt_type_max_prev,operaton_amt_desc_max_prev,operaton_amt_sub_max_prev,one,event_type_nm_prev,event_type_nm_log_count,event_desc_prev,event_desc_log_count,channel_indicator_type_prev,channel_indicator_type_log_count,channel_indicator_sub_type_prev,channel_indicator_sub_type_log_count,operaton_amt_prev,operaton_amt_log_count,currency_iso_cd_prev,currency_iso_cd_log_count,mcc_code_prev,mcc_code_log_count,pos_cd_prev,pos_cd_log_count,browser_language_prev,browser_language_log_count,timezone_prev,timezone_log_count,session_id_prev,session_id_log_count,operating_system_type_prev,operating_system_type_log_count,battery_prev,battery_log_count,device_system_version_prev,device_system_version_log_count,developer_tools_prev,developer_tools_log_count,phone_voip_call_state_prev,phone_voip_call_state_log_count,web_rdp_connection_prev,web_rdp_connection_log_count,compromised_prev,compromised_log_count,compromised_ses_count
i64,f64,i64,datetime[μs],i8,i16,i8,i8,f64,i8,i8,i8,bool,i32,i64,i8,bool,bool,i8,i8,i8,i8,bool,f32,f32,i16,i64,i64,f64,f64,f64,f64,f64,i32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,bool,f32,u16
123707242230467,0.000024,123123123123129,2025-07-27 14:55:11,8,122,2,0,-1.0,-1,-1,-1,true,-1,125597024902696,-1,true,true,-1,-1,-1,-1,false,-1.0,-1.0,-1,94053,null,1.10227007e8,1.10227007e8,-1.0,-1.0,-1.0,1,false,5.509388,false,4.356709,false,4.356709,false,4.356709,false,6.943122,false,6.910751,false,7.30586,true,7.732808,true,8.018296,true,7.878534,false,0.693147,true,8.018296,true,8.018296,true,7.86442,true,7.86442,true,7.908755,true,7.998672,true,7.824846,1
123234793229123,0.000233,123123123123129,2025-07-27 14:55:20,14,42,4,15,247750.0,0,-1,-1,true,-1,125597024902696,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,9,9,1.10227007e8,1.10227007e8,2.9739e7,2.012582e6,2.9739e7,1,false,7.466799,false,3.135494,false,6.855409,false,6.855409,false,0.693147,false,7.618251,true,7.306531,true,7.733246,true,8.018625,true,7.878913,true,1.098612,true,8.018625,true,8.018625,false,6.075346,false,6.075346,false,5.758902,true,7.999007,false,6.284134,2
125837545055907,0.000229,123123123123129,2025-07-27 14:55:28,7,56,4,15,-1.0,-1,-1,-1,true,-1,125828952579121,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,8,null,1.10227007e8,1.10227007e8,2.9739e7,-1.0,2.9739e7,1,false,6.763885,false,6.766191,true,6.856462,true,6.856462,false,6.944087,false,6.911747,true,7.307202,true,7.733684,true,8.018954,true,7.879292,false,0.693147,true,8.018954,true,8.018954,true,6.077642,true,6.077642,true,5.762052,true,7.999343,true,6.285998,1
126456020999239,0.000321,123123123123129,2025-07-27 14:56:28,7,56,4,15,-1.0,-1,-1,-1,true,-1,125597024902696,-1,true,false,0,0,-1,0,false,1080.0,2180.0,20,60,68,1.10227007e8,1.10227007e8,2.9739e7,-1.0,2.9739e7,1,true,6.765039,true,6.767343,true,6.857514,true,6.857514,true,6.945051,true,6.912743,true,7.307873,true,7.734121,true,8.019284,true,7.87967,false,1.386294,true,8.019284,true,8.019284,true,6.079933,true,6.079933,true,5.765191,true,7.999679,true,6.287858,3
125090221587312,0.000159,123123123123130,2025-07-09 06:42:00,8,86,4,15,496350.0,0,-1,-1,true,-1,125605614487041,-1,true,false,0,0,-1,0,false,720.0,1507.0,19,106964,null,6.5867429e7,6.5867429e7,6.45125e7,2.9715e6,6.45125e7,1,false,6.09131,false,6.070738,true,7.597396,true,7.548029,false,0.693147,true,7.578146,true,7.866722,true,7.952967,true,8.083637,true,7.932721,false,0.693147,true,8.083637,true,8.083637,true,6.656726,true,6.656726,true,6.448889,true

In [13]:
test0.select("event_id","predict").write_csv('submit009logf.csv')